In [1]:
!pip install -q openai-whisper
!pip install -q langdetect
!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import whisper
import gradio as gr
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
print("Loading Whisper model...")

whisper_model = whisper.load_model("base")

print("Whisper model loaded successfully!")

Loading Whisper model...


100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 61.3MiB/s]


Whisper model loaded successfully!


In [4]:
def speech_to_text(audio_file):

    result = whisper_model.transcribe(audio_file)

    transcript = result["text"].strip()

    return transcript

In [5]:
def detect_language(text):

    if not text or not text.strip():
        return "No text provided"

    try:
        language_code = detect(text)

        language_names = {
            "en": "English 🇬🇧",
            "ta": "Tamil 🇮🇳",
            "hi": "Hindi 🇮🇳",
            "te": "Telugu 🇮🇳",
            "ml": "Malayalam 🇮🇳",
            "kn": "Kannada 🇮🇳",
            "fr": "French 🇫🇷",
            "de": "German 🇩🇪",
            "es": "Spanish 🇪🇸",
            "it": "Italian 🇮🇹",
            "pt": "Portuguese 🇵🇹",
            "ar": "Arabic 🌍",
            "ja": "Japanese 🇯🇵",
            "ko": "Korean 🇰🇷",
            "zh-cn": "Chinese 🇨🇳"
        }

        return language_names.get(
            language_code,
            f"Unknown Language ({language_code})"
        )

    except Exception:
        return "Could not identify language"

In [6]:
def language_analyzer(audio_file, text_input):

    # Get text from speech or text input
    if audio_file is not None:
        transcript = speech_to_text(audio_file)

    elif text_input and text_input.strip():
        transcript = text_input.strip()

    else:
        return (
            "Please enter text or record/upload audio.",
            ""
        )

    if not transcript:
        return (
            "Could not understand the input.",
            ""
        )

    # Detect language
    language = detect_language(transcript)

    return (
        transcript,
        language
    )

In [7]:
print(language_analyzer(
    None,
    "Hello, how are you today?"
))

('Hello, how are you today?', 'Unknown Language (so)')


In [8]:
with gr.Blocks(title="Language Identification System") as app:

    gr.Markdown("""
    # 🌍 Language Identification System

    ### Identify the language of text or speech

    **Text / Speech → Whisper ASR → Language Detection → Result**
    """)

    with gr.Row():

        with gr.Column():

            text_input = gr.Textbox(
                label="📝 Enter Text",
                placeholder="Type a sentence in any language...",
                lines=6
            )

            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎤 Or Record / Upload Audio"
            )

            identify_button = gr.Button(
                "🌍 Identify Language",
                variant="primary"
            )

        with gr.Column():

            transcript_output = gr.Textbox(
                label="📝 Recognized / Entered Text",
                lines=6
            )

            language_output = gr.Textbox(
                label="🌍 Detected Language",
                lines=2
            )

    identify_button.click(
        fn=language_analyzer,
        inputs=[audio_input, text_input],
        outputs=[
            transcript_output,
            language_output
        ]
    )


app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://55e7fe9247408504e5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
